# Hand-to-Tex: End-to-End Pipeline Workflow
This notebook demonstrates how to load datasets, initialize the `BaselineTransformer`, and execute the `BaselineTrainer` loop step-by-step.

In [1]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import sys
from pathlib import Path

import torch

# Add the parent directory to the path so we can import the local hand_to_tex module perfectly
sys.path.append("../src")

from hand_to_tex.datasets.dataloader import HMEDataLoaderFactory
from hand_to_tex.models import BaselineTransformer
from hand_to_tex.training import BaselineTrainer
from hand_to_tex.utils import LatexVocab

# Force math on CPU for compatibility and debug, or mps/cuda if computationally available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"We are running isolated processes on: {device}")

We are running isolated processes on: mps


## 1. Vocabulary and DataLoaders
First, we load the `LatexVocab` to understand the size of our dictionary. Then we initialize our training and validation dataloaders pointing to the downloaded `mathwriting-2024` dataset.

*Note: Ensure you have run `uv run htt-get-data` in your terminal root so that the dataset exists!*

In [2]:
vocab = LatexVocab.load(Path("../data/assets/vocab.json"))
print(f"Loaded Vocabulary Dimension Limits: {len(vocab._encode)} items")

# Point to data folder in our base repository
root_path = Path("../data/mathwriting-2024-excerpt")

factory = HMEDataLoaderFactory(root=root_path, vocab=vocab, batch_size=32)
train_loader = factory.train()

try:
    valid_loader = factory.valid()
except Exception:
    print("No validation set found. Reusing train_loader for validation tests...")
    valid_loader = factory.train()

print(f"Batches established successfully: {len(train_loader)} separate batches.")

Loaded Vocabulary Dimension Limits: 258 items
Batches established successfully: 3 separate batches.


## 2. Model Initialization
We declare the `BaselineTransformer` with explicit model dimensions to process our geometric inputs.

In [3]:
model = BaselineTransformer(
    vocab_size=len(vocab._encode),
    pad_idx=vocab.PAD,
    d_model=64,  # Dimensional size of the abstract mathematical thought state
    nhead=2,  # Number of parallel attention heads running logic checks
    num_encoder_layers=2,  # Transformer blocks decoding the geometric physical handwriting
    num_decoder_layers=2,  # Transformer blocks translating geometry to math text sequence
    in_features=10,  # Initial parameters per point (speed, coordinates, etc.)
)

print(
    f"Model Trainable Constraint count: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters"
)

Model Trainable Constraint count: 666,562 parameters


## 3. Training Loop Execution
The `BaselineTrainer` encapsulates and automates all required gradient scaling, backpropagation, and target decoding evaluation processes natively.

In [4]:
trainer = BaselineTrainer(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    vocab=vocab,
    device=device,
    lr=5e-4,
)

# Start training for a few isolated epochs using the built infrastructure to track the loss dropping
trainer.train(num_epochs=10)

========== Training for 10 epochs ==========

[ Epoch1/10 ]


/Users/marcin/Documents/GitHub/hand-to-tex/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch completed in 37.67s
Train Loss: 5.3898  |  Valid Loss: 5.0842

[ Epoch2/10 ]
Epoch completed in 37.12s
Train Loss: 4.8736  |  Valid Loss: 4.9060

[ Epoch3/10 ]
Epoch completed in 36.95s
Train Loss: 4.7282  |  Valid Loss: 4.8096

[ Epoch4/10 ]
Epoch completed in 37.40s
Train Loss: 4.6243  |  Valid Loss: 4.7332

[ Epoch5/10 ]
Epoch completed in 36.95s
Train Loss: 4.5482  |  Valid Loss: 4.6671

[ Epoch6/10 ]
Epoch completed in 37.14s
Train Loss: 4.4856  |  Valid Loss: 4.6022

[ Epoch7/10 ]
Epoch completed in 37.03s
Train Loss: 4.3917  |  Valid Loss: 4.5262

[ Epoch8/10 ]
Epoch completed in 36.92s
Train Loss: 4.3364  |  Valid Loss: 4.4446

[ Epoch9/10 ]
Epoch completed in 36.96s
Train Loss: 4.2069  |  Valid Loss: 4.3799

[ Epoch10/10 ]
Epoch completed in 37.03s
Train Loss: 4.1294  |  Valid Loss: 4.3234
